# 02 - Protocol Adapter Demonstration

This notebook demonstrates how the DERIM middleware protocol adapters work.
We simulate a Modbus device using `pymodbus`'s built-in simulator and show
how the `ModbusAdapter` reads data and normalises it into the common
`DERTelemetry` model.

## Objectives

1. Understand the `BaseAdapter` interface.
2. Simulate a Modbus TCP server with sample register values.
3. Use the `ModbusAdapter` to connect, read data, and send commands.
4. Show how MQTT telemetry payloads are normalised.

---

In [ ]:
import sys
sys.path.insert(0, '../src')

from derim.models.common import (
    DERTelemetry, SolarTelemetry, BatteryTelemetry,
    EVChargerTelemetry, CommandRequest, DERType, DeviceState
)
from derim.models.adapters import ModbusRegisterBlock, MQTTMessage
from datetime import datetime, timezone
import json

## 1. Understanding the Common Data Model

All protocol adapters normalise their raw data into the `DERTelemetry` model.
Let's create some sample records to understand the schema.

In [ ]:
# Create a generic telemetry record
telemetry = DERTelemetry(
    device_id='solar-inv-001',
    device_type=DERType.SOLAR_PV,
    power_kw=4.75,
    energy_kwh=1234.5,
    voltage_v=230.1,
    current_a=20.6,
    frequency_hz=50.01,
    state=DeviceState.ON,
    power_factor=0.98,
)

print('=== DERTelemetry ===')
print(telemetry.model_dump_json(indent=2))

In [ ]:
# Solar-specific telemetry with irradiance and panel temperature
solar = SolarTelemetry(
    device_id='solar-inv-002',
    power_kw=3.2,
    energy_kwh=567.8,
    voltage_v=231.5,
    current_a=13.8,
    state=DeviceState.ON,
    irradiance_w_m2=850.0,
    panel_temperature_c=45.2,
    dc_voltage_v=380.5,
    dc_current_a=8.4,
)

print('=== SolarTelemetry ===')
print(solar.model_dump_json(indent=2))

In [ ]:
# Battery telemetry with state of charge
battery = BatteryTelemetry(
    device_id='bess-001',
    power_kw=-2.5,  # Negative = charging
    energy_kwh=100.0,
    voltage_v=48.2,
    current_a=52.0,
    state=DeviceState.ON,
    soc_percent=65.3,
    soh_percent=98.1,
    temperature_c=28.5,
    charge_rate_kw=2.5,
)

print('=== BatteryTelemetry ===')
print(battery.model_dump_json(indent=2))

## 2. Simulating Modbus Register Data

In a real deployment, the `ModbusAdapter` reads holding registers from a
physical device. Here we simulate what raw register data looks like and
how it maps to telemetry fields.

In [ ]:
# Simulate raw Modbus register block
register_block = ModbusRegisterBlock(
    unit_id=1,
    start_address=40071,
    values=[2060, 0, 0, 0, 0, 0, 0, 0, 2301, 0, 0, 0, 475, 5001, 0, 0, 0, 0, 0, 0, 980, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
)

print('=== Raw Modbus Register Block ===')
print(f'Unit ID: {register_block.unit_id}')
print(f'Start Address: {register_block.start_address}')
print(f'Number of registers: {len(register_block.values)}')
print(f'Timestamp: {register_block.timestamp}')

# Show how registers map to telemetry fields
from derim.adapters.modbus import DEFAULT_REGISTER_MAP

print('\n=== Default Register Map ===')
for name, (addr, count, scale, unit) in DEFAULT_REGISTER_MAP.items():
    print(f'  {name:20s} -> Address {addr}, Count {count}, Scale {scale}, Unit: {unit}')

## 3. Simulating MQTT Telemetry

MQTT adapters receive JSON payloads on subscribed topics. Here we show
the expected payload format and how it maps to `DERTelemetry`.

In [ ]:
# Simulate an MQTT message from a battery device
mqtt_payload = {
    'device_type': 'battery',
    'power_kw': -3.5,
    'energy_kwh': 250.0,
    'voltage_v': 48.1,
    'current_a': 72.8,
    'frequency_hz': 50.0,
    'state': 'on',
    'soc_percent': 42.5,
    'temperature_c': 31.2,
}

mqtt_msg = MQTTMessage(
    topic='derim/devices/bess-002/telemetry',
    payload=mqtt_payload,
    qos=1,
)

print('=== MQTT Message ===')
print(f'Topic: {mqtt_msg.topic}')
print(f'QoS: {mqtt_msg.qos}')
print(f'Payload:')
print(json.dumps(mqtt_msg.payload, indent=2))

# Normalise to DERTelemetry
normalised = DERTelemetry(
    device_id='bess-002',
    device_type=DERType.BATTERY,
    power_kw=mqtt_payload['power_kw'],
    energy_kwh=mqtt_payload['energy_kwh'],
    voltage_v=mqtt_payload['voltage_v'],
    current_a=mqtt_payload['current_a'],
    frequency_hz=mqtt_payload['frequency_hz'],
    state=DeviceState.ON,
    metadata={'protocol': 'mqtt', 'topic': mqtt_msg.topic},
)

print('\n=== Normalised DERTelemetry ===')
print(normalised.model_dump_json(indent=2))

## 4. Command Model

Control commands follow a standardised format regardless of the underlying protocol.

In [ ]:
# Create a setpoint command
cmd = CommandRequest(
    command='setpoint',
    value=3.5,
    parameters={'ramp_rate_kw_s': 0.5}
)

print('=== Control Command ===')
print(cmd.model_dump_json(indent=2))

# On/Off command
cmd_off = CommandRequest(command='off')
print('\n=== Off Command ===')
print(cmd_off.model_dump_json(indent=2))

## Summary

This notebook demonstrated:

- The common `DERTelemetry` data model and its device-specific extensions.
- How raw Modbus registers map to normalised telemetry fields.
- How MQTT JSON payloads are converted to the common model.
- The standardised `CommandRequest` format for device control.

For live adapter usage with actual devices, see the adapter documentation
and the `derim.adapters` package.